# 04b · Supporting Exploratory Figures

**Owner: Yogesh Basnet · Epic: Visualisation**

Three figures that defend claims the report makes, rather than decorate it.

1. **Data availability across seventeen years** — evidence for the coverage claim, and an honest
   picture of where the record is thin.
2. **Autocorrelation and partial autocorrelation** — turns the choice of lags 1, 2, 3, 24 and 168
   from an assertion into a justified design decision.
3. **Irradiance against power** — the core physical relationship the entire project rests on,
   with the temperature derating effect visible.

Runs from the stores written by notebooks 03 and 04.

In [ ]:
%matplotlib inline
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

REPO = Path("/Users/uttamshrestha/Desktop/Data Science Practice Unit/PRT661-solar-forecasting")
PROC, FIG = REPO/"Datasets"/"processed", REPO/"Outputs"/"figures"
FIG.mkdir(parents=True, exist_ok=True)
TARGET = "Active_Power"

h = pd.read_parquet(PROC/"mastermeter1_hourly_clean.parquet")
f = pd.read_parquet(PROC/"features_hourly.parquet")
plt.rcParams.update({"figure.dpi":130,"savefig.bbox":"tight","font.size":9,
                     "axes.grid":True,"grid.alpha":.25,
                     "axes.spines.top":False,"axes.spines.right":False})
print(f"hourly {len(h):,} rows | features {f.shape[1]} cols")

## 1 · Figure 4 · Data availability by month

Each cell is the proportion of that month's hours carrying a target value. Dark bands are outages, not seasonality.

In [ ]:
avail = (h[TARGET].notna().resample("MS").mean()).to_frame("frac")
avail["year"], avail["month"] = avail.index.year, avail.index.month
grid = avail.pivot_table(index="year", columns="month", values="frac")

fig, ax = plt.subplots(figsize=(8, 0.32*len(grid)+1.4))
im = ax.imshow(grid.values, aspect="auto", cmap="YlGnBu", vmin=0, vmax=1)
ax.set_xticks(range(12)); ax.set_xticklabels(list("JFMAMJJASOND"))
ax.set_yticks(range(len(grid))); ax.set_yticklabels(grid.index)
ax.set_title("Proportion of hours with a usable target value")
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=.7, label="fraction complete")
fig.tight_layout(); fig.savefig(FIG/"fig4_data_availability.png")
plt.show()

worst = grid.stack().sort_values().head(8)
print("least complete months:")
for (y, m), v in worst.items():
    print(f"   {int(y)}-{int(m):02d}  {100*v:5.1f}%")
print(f"\noverall completeness: {100*h[TARGET].notna().mean():.2f}%")
print("saved fig4_data_availability.png")

## 2 · Figure 5 · Autocorrelation on the hourly grid

Computed on the **complete hourly series**, including night, using a time aware shift.

An earlier version of this figure correlated by position within a daylight-only series. With
roughly 12 daylight hours per day, position 24 in that series is about two days, so the axis
labelled hours was not hours. Excluding night is correct when cross-correlating with features,
where a shared overnight zero inflates the relationship (figure 3). It is wrong for
autocorrelation, where the daily cycle *is* the signal and deleting half of every day destroys
the time axis.

`Series.corr(Series.shift(k))` aligns on the DatetimeIndex and drops missing pairs, so lag k is
exactly k hours. The partial autocorrelation is what justifies the lag set: it shows which lags
carry information the shorter lags do not already contain.

In [ ]:
s = h[TARGET]                     # full hourly grid, night included
MAXLAG, PMAX = 200, 48
N = int(s.notna().sum())

acf = np.array([1.0] + [s.corr(s.shift(k)) for k in range(1, MAXLAG + 1)])

pacf = [1.0]                      # Durbin-Levinson
for k in range(1, PMAX + 1):
    R = np.array([[acf[abs(i - j)] for j in range(k)] for i in range(k)])
    try:
        pacf.append(np.linalg.solve(R, acf[1:k + 1])[-1])
    except np.linalg.LinAlgError:
        pacf.append(np.nan)
pacf = np.array(pacf)

ci = 1.96 / np.sqrt(N)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].bar(range(len(acf)), acf, color="#1f77b4", width=.9)
for L in (24, 48, 72, 168):
    if L <= MAXLAG:
        ax[0].axvline(L, color="#999", lw=.6, ls=":")
ax[0].axhline(ci, ls="--", c="#d62728", lw=.8); ax[0].axhline(-ci, ls="--", c="#d62728", lw=.8)
ax[0].set_title(f"ACF to {MAXLAG} hours"); ax[0].set_xlabel("lag (hours)")

ax[1].bar(range(len(pacf)), pacf, color="#2ca02c", width=.7)
for L in (1, 24, 48):
    if L <= PMAX:
        ax[1].axvline(L, color="#999", lw=.6, ls=":")
ax[1].axhline(ci, ls="--", c="#d62728", lw=.8); ax[1].axhline(-ci, ls="--", c="#d62728", lw=.8)
ax[1].set_title(f"PACF to {PMAX} hours"); ax[1].set_xlabel("lag (hours)")
fig.tight_layout(); fig.savefig(FIG/"fig5_autocorrelation.png")
plt.show()

print(f"n = {N:,} hourly observations, 95% band = +/- {ci:.4f}\n")
print("ACF at the chosen lags:")
for L in [1, 2, 3, 24, 168]:
    print(f"   lag {L:>3}h : {acf[L]:+.3f}")

peaks = [L for L in range(2, MAXLAG) if acf[L] > acf[L-1] and acf[L] > acf[L+1] and acf[L] > 0.2]
print(f"\nACF local maxima above 0.2 (expect multiples of 24 if the daily cycle dominates):")
print("   ", peaks[:10])

print("\nlargest PACF beyond lag 3:")
tail = pd.Series(pacf[4:], index=range(4, len(pacf)))
print(tail.abs().sort_values(ascending=False).head(6).round(3).to_string())

## 3 · Figure 6 · Irradiance against power, and temperature derating

The left panel is the physical transfer curve. The right panel bins by module relevant temperature: photovoltaic efficiency falls as cells heat, so at equal irradiance hotter hours should yield slightly less power.

In [ ]:
d = f[(f["is_daylight"] == 1)].dropna(
        subset=["y", "Global_Horizontal_Radiation", "Weather_Temperature_Celsius"])
samp = d.sample(min(30000, len(d)), random_state=0)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
sc = ax[0].scatter(samp["Global_Horizontal_Radiation"], samp["y"], s=2, alpha=.18,
                   c=samp["Weather_Temperature_Celsius"], cmap="coolwarm")
ax[0].set_xlabel("Global horizontal radiation (W/m2)"); ax[0].set_ylabel("Active Power (kW)")
ax[0].set_title("Transfer curve, coloured by air temperature")
fig.colorbar(sc, ax=ax[0], shrink=.8, label="deg C")

bins = [0, 15, 25, 32, 38, 60]
labels = ["<15", "15-25", "25-32", "32-38", ">38"]
d = d.assign(tband=pd.cut(d["Weather_Temperature_Celsius"], bins, labels=labels),
             gband=pd.cut(d["Global_Horizontal_Radiation"], np.arange(0, 1300, 100)))
prof = d.groupby(["gband", "tband"], observed=True)["y"].mean().unstack()
for c, col in zip(prof.columns, ["#2c7bb6","#abd9e9","#ffffbf","#fdae61","#d7191c"]):
    ax[1].plot([iv.mid for iv in prof.index], prof[c].values, "o-", ms=3, lw=1.3,
               color=col, label=f"{c} C")
ax[1].set_xlabel("Global horizontal radiation (W/m2)"); ax[1].set_ylabel("mean Active Power (kW)")
ax[1].set_title("Mean power by irradiance and temperature band")
ax[1].legend(frameon=False, fontsize=7, title="air temp", title_fontsize=7)
fig.tight_layout(); fig.savefig(FIG/"fig6_irradiance_power.png")
plt.show()

r = d[["Global_Horizontal_Radiation", "y"]].corr().iloc[0,1]
print(f"Pearson r, GHI vs Active Power (daylight): {r:.4f}")
print(f"\nmean power at 700-800 W/m2 by temperature band:")
row = prof.loc[[iv for iv in prof.index if iv.mid == 750][0]]
print(row.round(2).to_string())
hi, lo = row.dropna().index[-1], row.dropna().index[0]
print(f"\ndifference {lo} C vs {hi} C at equal irradiance: "
      f"{row[hi]-row[lo]:+.2f} kW ({100*(row[hi]-row[lo])/row[lo]:+.1f}%)")
print("saved fig6_irradiance_power.png")